In [1]:
!pip install requests beautifulsoup4 pandas plotly wordcloud lxml -q
print("Libraries ready ✅")

Libraries ready ✅


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def scrape_yearly_boxoffice(year):
    url = f"https://www.boxofficemojo.com/year/{year}/"
    print(f"Scraping {year}...")

    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            print(f"Blocked for {year} — status {response.status_code}")
            return None

        soup = BeautifulSoup(response.content, 'lxml')
        table = soup.find('table')

        if not table:
            print(f"No table found for {year}")
            return None

        rows = []
        for tr in table.find_all('tr')[1:]:
            cols = [td.get_text(strip=True) for td in tr.find_all('td')]
            if cols:
                rows.append(cols)

        headers_row = [th.get_text(strip=True) for th in table.find_all('th')]
        df = pd.DataFrame(rows, columns=headers_row[:len(rows[0])] if headers_row else None)
        df['year'] = year
        return df

    except Exception as e:
        print(f"Error scraping {year}: {e}")
        return None

# Scrape 5 years
years = [2019, 2020, 2021, 2022, 2023]
all_data = []

for year in years:
    df_year = scrape_yearly_boxoffice(year)
    if df_year is not None:
        all_data.append(df_year)
    time.sleep(2)  # polite delay between requests

if all_data:
    raw_df = pd.concat(all_data, ignore_index=True)
    print(f"\nRaw data shape: {raw_df.shape}")
    print(raw_df.head())
    raw_df.to_csv("raw_boxoffice.csv", index=False)
    print("Raw data saved ✅")
else:
    print("Scraping failed — switching to backup plan")

Scraping 2019...
Scraping 2020...
Scraping 2021...
Scraping 2022...
Scraping 2023...

Raw data shape: (1000, 12)
  Rank            Release Genre Budget Running Time         Gross Theaters  \
0    1  Avengers: Endgame     -      -            -  $858,373,000    4,662   
1    2      The Lion King     -      -            -  $543,638,043    4,802   
2    3        Toy Story 4     -      -            -  $434,038,008    4,575   
3    4          Frozen II     -      -            -  $430,144,682    4,440   
4    5     Captain Marvel     -      -            -  $426,829,839    4,310   

    Total Gross Release Date                          Distributor Estimated  \
0  $858,373,000       Apr 26  Walt Disney Studios Motion Pictures     false   
1  $543,638,043       Jul 19  Walt Disney Studios Motion Pictures     false   
2  $434,038,008       Jun 21  Walt Disney Studios Motion Pictures     false   
3  $477,373,578       Nov 22  Walt Disney Studios Motion Pictures     false   
4  $426,829,839        

In [3]:
import numpy as np

df = raw_df.copy()

# Step 3a — Rename columns cleanly
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Columns:", df.columns.tolist())

# Step 3b — Clean money columns (remove $ and ,)
def clean_money(val):
    if pd.isna(val) or val in ['-', '', 'N/A']:
        return np.nan
    return float(str(val).replace('$','').replace(',','').strip())

df['gross']       = df['gross'].apply(clean_money)
df['total_gross'] = df['total_gross'].apply(clean_money)

# Step 3c — Clean theaters column
df['theaters'] = pd.to_numeric(
    df['theaters'].astype(str).str.replace(',','').str.strip(),
    errors='coerce'
)

# Step 3d — Fix release date (add year)
df['release_date'] = df['release_date'].astype(str).str.strip()
df['release_date_full'] = df['release_date'] + ' ' + df['year'].astype(str)
df['release_date_full'] = pd.to_datetime(
    df['release_date_full'], format='%b %d %Y', errors='coerce'
)
df['month'] = df['release_date_full'].dt.month
df['month_name'] = df['release_date_full'].dt.strftime('%b')

# Step 3e — Replace dashes with NaN
df.replace('-', np.nan, inplace=True)

# Step 3f — Drop duplicates
before = len(df)
df.drop_duplicates(subset=['release', 'year'], inplace=True)
after = len(df)
print(f"Removed {before - after} duplicates")

# Step 3g — Add performance tier based on total gross
def performance_tier(gross):
    if pd.isna(gross):
        return 'Unknown'
    elif gross >= 300_000_000:
        return 'Blockbuster'
    elif gross >= 100_000_000:
        return 'Hit'
    elif gross >= 50_000_000:
        return 'Average'
    else:
        return 'Underperformer'

df['performance_tier'] = df['total_gross'].apply(performance_tier)

# Step 3h — Add distributor clean column
df['distributor'] = df['distributor'].str.strip()
df['distributor'] = df['distributor'].fillna('Independent')

print(f"\nClean data shape: {df.shape}")
print(df['performance_tier'].value_counts())
print(df['year'].value_counts().sort_index())
print("\nMissing values:")
print(df.isnull().sum())
print("\nCleaning done ✅")

Columns: ['rank', 'release', 'genre', 'budget', 'running_time', 'gross', 'theaters', 'total_gross', 'release_date', 'distributor', 'estimated', 'year']
Removed 0 duplicates

Clean data shape: (1000, 16)
performance_tier
Underperformer    781
Average            97
Hit                90
Blockbuster        32
Name: count, dtype: int64
year
2019    200
2020    200
2021    200
2022    200
2023    200
Name: count, dtype: int64

Missing values:
rank                    0
release                 0
genre                1000
budget               1000
running_time         1000
gross                   0
theaters                8
total_gross             0
release_date            2
distributor             0
estimated               0
year                    0
release_date_full       2
month                   2
month_name              2
performance_tier        0
dtype: int64

Cleaning done ✅


/tmp/ipykernel_23762/1642704801.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('-', np.nan, inplace=True)


In [4]:
df.to_csv("clean_boxoffice_master.csv", index=False)
print("Master dataset saved ✅")
print(df.head())

Master dataset saved ✅
  rank            release  genre  budget  running_time        gross  theaters  \
0    1  Avengers: Endgame    NaN     NaN           NaN  858373000.0    4662.0   
1    2      The Lion King    NaN     NaN           NaN  543638043.0    4802.0   
2    3        Toy Story 4    NaN     NaN           NaN  434038008.0    4575.0   
3    4          Frozen II    NaN     NaN           NaN  430144682.0    4440.0   
4    5     Captain Marvel    NaN     NaN           NaN  426829839.0    4310.0   

   total_gross release_date                          distributor estimated  \
0  858373000.0       Apr 26  Walt Disney Studios Motion Pictures     false   
1  543638043.0       Jul 19  Walt Disney Studios Motion Pictures     false   
2  434038008.0       Jun 21  Walt Disney Studios Motion Pictures     false   
3  477373578.0       Nov 22  Walt Disney Studios Motion Pictures     false   
4  426829839.0        Mar 8  Walt Disney Studios Motion Pictures     false   

   year release_date_

In [6]:
import plotly.express as px
import plotly.graph_objects as go
import base64
from io import BytesIO
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# ── helpers ──────────────────────────────────────────────────────────────────
def money(n):
    if n >= 1_000_000_000:
        return f"${n/1_000_000_000:.1f}B"
    elif n >= 1_000_000:
        return f"${n/1_000_000:.0f}M"
    return f"${n:,.0f}"

def wc_to_base64(text, colormap="viridis"):
    wc = WordCloud(width=800, height=300,
                   background_color="#1C1C1E",
                   colormap=colormap).generate(text)
    buf = BytesIO()
    wc.to_image().save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

# ── KPI values ────────────────────────────────────────────────────────────────
total_films      = len(df)
total_gross      = df['total_gross'].sum()
avg_gross        = df['total_gross'].mean()
top_film         = df.loc[df['total_gross'].idxmax(), 'release']
blockbusters     = len(df[df['performance_tier'] == 'Blockbuster'])

# ── Chart 1: Gross by year ────────────────────────────────────────────────────
yearly = df.groupby('year')['total_gross'].sum().reset_index()
yearly['gross_label'] = yearly['total_gross'].apply(money)
fig1 = px.bar(yearly, x='year', y='total_gross',
              text='gross_label',
              title='Total Box Office Gross by Year',
              color='total_gross',
              color_continuous_scale=['#E23744','#FF6B7A'])
fig1.update_traces(textposition='outside')
fig1.update_layout(paper_bgcolor='rgba(0,0,0,0)',
                   plot_bgcolor='rgba(0,0,0,0)',
                   font_color='#ffffff',
                   coloraxis_showscale=False)

# ── Chart 2: Performance tier breakdown ──────────────────────────────────────
tier_colors = {
    'Blockbuster': '#E23744',
    'Hit':         '#F39C12',
    'Average':     '#95A5A6',
    'Underperformer': '#2C2C2E'
}
tier_counts = df['performance_tier'].value_counts().reset_index()
tier_counts.columns = ['tier', 'count']
fig2 = px.pie(tier_counts, names='tier', values='count',
              title='Performance Tier Distribution',
              color='tier', color_discrete_map=tier_colors,
              hole=0.4)
fig2.update_layout(paper_bgcolor='rgba(0,0,0,0)',
                   font_color='#ffffff')

# ── Chart 3: Top 10 films by gross ───────────────────────────────────────────
top10 = df.nlargest(10, 'total_gross')[['release','total_gross','year']].copy()
top10['label'] = top10['total_gross'].apply(money)
fig3 = px.bar(top10, x='total_gross', y='release',
              orientation='h', text='label',
              title='Top 10 Highest Grossing Films (2019–2023)',
              color='total_gross',
              color_continuous_scale=['#FF6B7A','#E23744'])
fig3.update_traces(textposition='outside')
fig3.update_layout(paper_bgcolor='rgba(0,0,0,0)',
                   plot_bgcolor='rgba(0,0,0,0)',
                   font_color='#ffffff',
                   yaxis={'categoryorder':'total ascending'},
                   coloraxis_showscale=False)

# ── Chart 4: Monthly release pattern ─────────────────────────────────────────
month_order = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('month_name').size().reset_index(name='count')
monthly['month_name'] = pd.Categorical(
    monthly['month_name'], categories=month_order, ordered=True)
monthly = monthly.sort_values('month_name')
fig4 = px.line(monthly, x='month_name', y='count',
               title='Number of Releases by Month',
               markers=True,
               color_discrete_sequence=['#E23744'])
fig4.update_layout(paper_bgcolor='rgba(0,0,0,0)',
                   plot_bgcolor='rgba(0,0,0,0)',
                   font_color='#ffffff')

# ── Chart 5: Top distributors ────────────────────────────────────────────────
top_dist = df.groupby('distributor')['total_gross'].sum().nlargest(8).reset_index()
top_dist['label'] = top_dist['total_gross'].apply(money)
fig5 = px.bar(top_dist, x='distributor', y='total_gross',
              text='label', title='Top 8 Distributors by Total Gross',
              color='total_gross',
              color_continuous_scale=['#FF6B7A','#E23744'])
fig5.update_traces(textposition='outside')
fig5.update_layout(paper_bgcolor='rgba(0,0,0,0)',
                   plot_bgcolor='rgba(0,0,0,0)',
                   font_color='#ffffff',
                   xaxis_tickangle=-30,
                   coloraxis_showscale=False)

# ── Chart 6: Yearly performance tier stacked bar ─────────────────────────────
tier_year = df.groupby(['year','performance_tier']).size().reset_index(name='count')
fig6 = px.bar(tier_year, x='year', y='count',
              color='performance_tier',
              title='Performance Tiers by Year',
              color_discrete_map=tier_colors,
              barmode='stack')
fig6.update_layout(paper_bgcolor='rgba(0,0,0,0)',
                   plot_bgcolor='rgba(0,0,0,0)',
                   font_color='#ffffff')

# ── Word cloud of film titles ─────────────────────────────────────────────────
wc_img = wc_to_base64(" ".join(df['release'].dropna()), "Reds")

# ── Convert charts to HTML ────────────────────────────────────────────────────
c1 = fig1.to_html(full_html=False, include_plotlyjs=False)
c2 = fig2.to_html(full_html=False, include_plotlyjs=False)
c3 = fig3.to_html(full_html=False, include_plotlyjs=False)
c4 = fig4.to_html(full_html=False, include_plotlyjs=False)
c5 = fig5.to_html(full_html=False, include_plotlyjs=False)
c6 = fig6.to_html(full_html=False, include_plotlyjs=False)

# ── Build HTML dashboard ──────────────────────────────────────────────────────
html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>Film Box Office Analytics Dashboard</title>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ font-family:'Segoe UI',sans-serif;
          background:#1C1C1E; color:#ffffff; }}

  header {{
    background: linear-gradient(135deg,#E23744,#8B0000);
    padding:30px 40px;
  }}
  header h1 {{ font-size:1.9rem; font-weight:700; }}
  header p  {{ font-size:0.95rem; opacity:0.8; margin-top:6px; }}

  .kpi-row {{
    display:grid; grid-template-columns:repeat(5,1fr);
    gap:16px; padding:24px 40px 0;
  }}
  .kpi {{
    background:#2C2C2E; border-radius:12px;
    padding:18px 20px; text-align:center;
    border-top:3px solid #E23744;
  }}
  .kpi .num {{ font-size:1.6rem; font-weight:700; color:#E23744; }}
  .kpi .lbl {{ font-size:0.8rem; color:#aaa; margin-top:4px; }}

  .grid-2 {{
    display:grid; grid-template-columns:1fr 1fr;
    gap:16px; padding:24px 40px 0;
  }}
  .grid-1 {{
    padding:16px 40px 0;
  }}
  .card {{
    background:#2C2C2E; border-radius:12px;
    padding:20px;
  }}
  .card h3 {{ font-size:1rem; color:#aaa;
              margin-bottom:10px; }}
  .card img {{ width:100%; border-radius:8px; }}

  footer {{
    text-align:center; padding:24px;
    font-size:0.8rem; color:#555;
    margin-top:24px;
  }}
</style>
</head>
<body>

<header>
  <h1>🎬 Film Box Office Intelligence Dashboard</h1>
  <p>Analysing {total_films:,} films across 2019–2023 · ETL Pipeline built with Python & BeautifulSoup</p>
</header>

<!-- KPI Row -->
<div class="kpi-row">
  <div class="kpi">
    <div class="num">{total_films:,}</div>
    <div class="lbl">Total Films</div>
  </div>
  <div class="kpi">
    <div class="num">{money(total_gross)}</div>
    <div class="lbl">Total Box Office</div>
  </div>
  <div class="kpi">
    <div class="num">{money(avg_gross)}</div>
    <div class="lbl">Avg Gross per Film</div>
  </div>
  <div class="kpi">
    <div class="num">{blockbusters}</div>
    <div class="lbl">Blockbusters</div>
  </div>
  <div class="kpi">
    <div class="num" style="font-size:0.95rem">{top_film[:20]}..</div>
    <div class="lbl">Highest Grossing Film</div>
  </div>
</div>

<!-- Row 1 -->
<div class="grid-2">
  <div class="card">{c1}</div>
  <div class="card">{c2}</div>
</div>

<!-- Row 2 -->
<div class="grid-1">
  <div class="card">{c3}</div>
</div>

<!-- Row 3 -->
<div class="grid-2">
  <div class="card">{c4}</div>
  <div class="card">{c6}</div>
</div>

<!-- Row 4 -->
<div class="grid-1">
  <div class="card">{c5}</div>
</div>

<!-- Word Cloud -->
<div class="grid-1">
  <div class="card">
    <h3>🎬 Film Titles Word Cloud (2019–2023)</h3>
    <img src="data:image/png;base64,{wc_img}"/>
  </div>
</div>

<footer>
  Built by Varshini · Film Box Office ETL Pipeline · Data Source: Box Office Mojo · 2026
</footer>

</body>
</html>
"""

with open("film_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html)

from google.colab import files
files.download("film_dashboard.html")

print("Dashboard downloaded ✅ Open in browser!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Dashboard downloaded ✅ Open in browser!
